In [61]:
from src.cyk import CYK
import pandas as pd
import numpy as np
import spacy


# Manual (POS) tagging and cyk Tree

In [62]:
data = pd.read_csv("data/sentences.csv", sep=";")
G = [
        # S is axiom
        ("S", ("NP", "VP")), 
        
            # non terminal rules
        ("NP", ("DET", "NOUN")),
        ("PP", ("ADP", "NP")),
        ("VP", ("VERB", "PP")),

        
        # terminal (if using pos directly)
        ("DET", ("DET",)),
        ("VERB", ("VERB",)),
        ("NOUN", ("NOUN",)),
        ("ADP", ("ADP",))
    ]
    
cyk = CYK(G)

for _, example in data.iterrows():
    sentence = list(example['Sentence'].split())
    list_pos = []
    for p in example['Pos'].split():
        list_pos.append([p])
    print(f"Sentence: {sentence}")
    print(f"Pos: {list_pos}")
    
    tree = cyk(list_pos, sentence)
    for t in tree:
        print(t)
    print("##################################################################################")

    

Sentence: ['The', 'cat', 'sat', 'on', 'the', 'couch']
Pos: [['DET'], ['NOUN'], ['VERB'], ['ADP'], ['DET'], ['NOUN']]
S
----NP
--------DET - The
--------NOUN - cat
----VP
--------VERB - sat
--------PP
------------ADP - on
------------NP
----------------DET - the
----------------NOUN - couch


##################################################################################
Sentence: ['Time', 'flies', 'like', 'an', 'arrow']
Pos: [['NOUN'], ['VERB'], ['ADP'], ['DET'], ['NOUN']]
##################################################################################
Sentence: ['The', 'spy', 'saw', 'the', 'cop', 'with', 'the', 'telescope']
Pos: [['DET'], ['NOUN'], ['VERB'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']]
##################################################################################
Sentence: ['The', 'spy', 'saw', 'the', 'cop', 'with', 'the', 'revolver']
Pos: [['DET'], ['NOUN'], ['VERB'], ['DET'], ['NOUN'], ['ADP'], ['DET'], ['NOUN']]
##########################################

# Automatic (POS) tagging

## spaCy Model and sciSpaCy
- Having problems with the tagging

* **`en_core_web_sm`**: Small English model trained on **general-domain text**. This spaCy model is **not specialized for scientific or biomedical text**.

* **`en_core_sci_sm`**: This scispaCy model is **specialized for scientific or biomedical text**.




In [63]:
nlp_general = spacy.load("en_core_web_sm")
nlp_sci = spacy.load("en_core_sci_sm")

dict_sentences_web_sm = {}
dict_sentences_sci_sm = {}
for sentence in data['Sentence'].to_list():
    doc_general = nlp_general(sentence)
    doc_sci = nlp_sci(sentence)
    print(f"Sentence: {sentence}")

    pos_list_general = [token.pos_ for token in doc_general]
    print(f"Pos: {pos_list_general}")
    print(f'The len of pos_list_general: {len(pos_list_general)}')
    print("--------------------------------------------")
    dict_sentences_web_sm[sentence] = pos_list_general

    pos_list_sci = [token.pos_ for token in doc_sci]
    print(f"Pos: {pos_list_sci}")
    print(f'The len of pos_list_sci: {len(pos_list_sci)}')
    print("##################################################################################")
    dict_sentences_sci_sm[sentence] = pos_list_sci




Sentence: The cat sat on the couch
Pos: ['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
The len of pos_list_general: 6
--------------------------------------------
Pos: ['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
The len of pos_list_sci: 6
##################################################################################
Sentence: Time flies like an arrow
Pos: ['NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
The len of pos_list_general: 5
--------------------------------------------
Pos: ['NOUN', 'NOUN', 'ADP', 'DET', 'NOUN']
The len of pos_list_sci: 5
##################################################################################
Sentence: The spy saw the cop with the telescope
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
The len of pos_list_general: 8
--------------------------------------------
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
The len of pos_list_sci: 8
##################################################################################
S

In [64]:
dict_golden = {}
for _, example in data.iterrows():
    sentence = example['Sentence']
    pos_list = example['Pos'].split()
    dict_golden[sentence] = pos_list
    print(f"Sentence: {sentence}")
    print(f"Pos: {pos_list}")
    print(f"The len of pos_list: {len(pos_list)}")
    print("##################################################################################")


Sentence: The cat sat on the couch
Pos: ['DET', 'NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
The len of pos_list: 6
##################################################################################
Sentence: Time flies like an arrow
Pos: ['NOUN', 'VERB', 'ADP', 'DET', 'NOUN']
The len of pos_list: 5
##################################################################################
Sentence: The spy saw the cop with the telescope
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
The len of pos_list: 8
##################################################################################
Sentence: The spy saw the cop with the revolver
Pos: ['DET', 'NOUN', 'VERB', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN']
The len of pos_list: 8
##################################################################################
Sentence: Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature which is defined as thermomorphogenesis
Pos: ['PROPN', 'PROPN', 

## Evaluation of POS Tagging Performance in a Context-Free Environment

In [69]:
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score,f1_score


class EvaluatePOS:

    def __init__(self, dict_sentences, dict_golden):
        self.dict_sentences = dict_sentences
        self.dict_golden = dict_golden

    def evaluate(self):

        results = []

        all_predictions = []
        all_golden = []

        overall_errors = Counter()

        for sentence, pos_list in self.dict_sentences.items():

            # Sentence not found
            if sentence not in self.dict_golden:

                print(f"Warning: sentence not found:\n{sentence}\n")

                results.append({
                    "Sentence": sentence,
                    "Tokens": len(pos_list),
                    "Accuracy": "Not available",
                    "Precision": "Not available",
                    "Errors": "Not available"
                })

                continue

            golden_pos_list = self.dict_golden[sentence]

            # Length mismatch
            if len(pos_list) != len(golden_pos_list):

                print(
                    f"Warning: length mismatch:\n"
                    f"{sentence}\n"
                    f"Predicted tokens: {len(pos_list)}\n"
                    f"Golden tokens:    {len(golden_pos_list)}\n"
                )

                results.append({
                    "Sentence": sentence,
                    "Tokens": len(golden_pos_list),
                    "Accuracy": "Not available",
                    "Precision": "Not available",
                    "Errors": "Not available"
                })

                continue

            # Metrics
            accuracy = accuracy_score(
                golden_pos_list,
                pos_list
            )

            precision = precision_score(
                golden_pos_list,
                pos_list,
                average="macro",
                zero_division=0
            )

            f1 = f1_score(
                golden_pos_list,
                pos_list,
                average="macro",
                zero_division=0
            )

            # Find errors
            sentence_errors = Counter()

            for golden, predicted in zip(golden_pos_list, pos_list):

                if golden != predicted:

                    error = f"{golden} → {predicted}"

                    sentence_errors[error] += 1
                    overall_errors[error] += 1

            # Format errors
            if sentence_errors:
                errors_text = ", ".join(
                    f"{error} ({count})"
                    for error, count in sentence_errors.most_common()
                )
            else:
                errors_text = "None"

            results.append({
                "Sentence": sentence,
                "Tokens": len(golden_pos_list),
                "Accuracy": accuracy,
                "F1_macro": f1,
                "Precision_macro": precision,
                "Errors": errors_text
            })

            all_predictions.extend(pos_list)
            all_golden.extend(golden_pos_list)

        # Overall
        if all_golden:

            overall_accuracy = accuracy_score(
                all_golden,
                all_predictions
            )

            overall_f1 = f1_score(
                all_golden,
                all_predictions,
                average="macro",
                zero_division=0
            )
            
            overall_precision = precision_score(
                all_golden,
                all_predictions,
                average="macro",
                zero_division=0
            )

            if overall_errors:

                overall_errors_text = ", ".join(
                    f"{error} ({count})"
                    for error, count in overall_errors.most_common()
                )

            else:
                overall_errors_text = "None"

            results.append({
                "Sentence": "OVERALL",
                "Tokens": len(all_golden),
                "Accuracy": overall_accuracy,
                "F1_macro": overall_f1,
                "Precision_macro": overall_precision,
                "Errors": overall_errors_text
            })

        return pd.DataFrame(results)


In [70]:
web_sm_results = EvaluatePOS(dict_sentences_web_sm, dict_golden).evaluate()
web_sm_results.to_csv("results/POS_web_sm.csv", index=False)
web_sm_results



,Sentence,Tokens,Accuracy,F1_macro,Precision_macro,Errors
0,The cat sat on the couch,6,1.000000,1.000000,1.000000,None
1,Time flies like an arrow,5,1.000000,1.000000,1.000000,None
2,The spy saw the cop with the telescope,8,1.000000,1.000000,1.000000,None
3,The spy saw the cop with the revolver,8,1.000000,1.000000,1.000000,None
4,Arabidopsis thaliana seedlings exhibit longer ...,19,0.789474,0.703704,0.696296,"PROPN → NOUN (2), ADJ → ADV (1), NOUN → ADJ (1)"
5,A spectrogram of PSN J10354824 + 3900279 obtai...,23,0.956522,0.818182,0.818182,PUNCT → SYM (1)
6,OVERALL,69,0.927536,0.724532,0.735493,"PROPN → NOUN (2), ADJ → ADV (1), NOUN → ADJ (1..."


In [71]:
sci_sm_results = EvaluatePOS(dict_sentences_sci_sm, dict_golden).evaluate()
sci_sm_results.to_csv("results/POS_sci_sm.csv", index=False)
sci_sm_results

,Sentence,Tokens,Accuracy,F1_macro,Precision_macro,Errors
0,The cat sat on the couch,6,1.000000,1.000000,1.000000,None
1,Time flies like an arrow,5,0.800000,0.700000,0.666667,VERB → NOUN (1)
2,The spy saw the cop with the telescope,8,1.000000,1.000000,1.000000,None
3,The spy saw the cop with the revolver,8,1.000000,1.000000,1.000000,None
4,Arabidopsis thaliana seedlings exhibit longer ...,19,0.736842,0.600000,0.626667,"PROPN → NOUN (2), ADJ → ADV (1), NOUN → ADJ (1..."
5,A spectrogram of PSN J10354824 + 3900279 obtai...,23,0.782609,0.645336,0.658009,"PROPN → NOUN (3), PUNCT → CCONJ (1), PRON → DE..."
6,OVERALL,69,0.840580,0.638325,0.711689,"PROPN → NOUN (5), PRON → DET (2), VERB → NOUN ..."
